<a href="https://colab.research.google.com/github/Rishwantthi/CCTV-Based-Person-Detection-Timestamp-Identification-System/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 1. IMPORTS
# =========================
import tensorflow as tf
import numpy as np
import cv2
import os
import random
import zipfile
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder


# =========================
# 2. UPLOAD + UNZIP DATASET
# =========================
from google.colab import files
uploaded = files.upload()

zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('/content/train')


# =========================
# 3. DATASET PATH
# =========================
dataset_path = "/content/train/train"
print("Dataset folders:", os.listdir(dataset_path))


# =========================
# 4. LOAD DATASET
# =========================
data = []
labels = []

IMG_SIZE = 160

for person in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person)

    if not os.path.isdir(person_path):
        continue

    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0

        data.append(img)
        labels.append(person)

data = np.array(data)
labels = np.array(labels)


# =========================
# 5. ENCODE LABELS
# =========================
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)


# =========================
# 6. BUILD MODEL
# =========================
base_model = MobileNetV2(
    input_shape=(160,160,3),
    include_top=False,
    weights='imagenet'
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)

output = Dense(len(le.classes_), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(data, labels_encoded, epochs=10, batch_size=32)


# =========================
# 7. EMBEDDING MODEL
# =========================
embedding_model = Model(
    inputs=model.input,
    outputs=model.layers[-2].output
)


# =========================
# 8. GET EMBEDDING
# =========================
def get_embedding(img_path):
    img = cv2.imread(img_path)

    if img is None:
        return None

    img = cv2.resize(img, (160,160))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    emb = embedding_model.predict(img)[0]
    emb = emb / np.linalg.norm(emb)

    return emb


# =========================
# 9. AUTO THRESHOLD
# =========================
same_distances = []
diff_distances = []

people = os.listdir(dataset_path)

# SAME PERSON
for person in people:
    images = os.listdir(os.path.join(dataset_path, person))

    for i in range(min(5, len(images)-1)):
        img1 = os.path.join(dataset_path, person, images[i])
        img2 = os.path.join(dataset_path, person, images[i+1])

        emb1 = get_embedding(img1)
        emb2 = get_embedding(img2)

        same_distances.append(np.linalg.norm(emb1 - emb2))

# DIFFERENT PERSON
for i in range(5):
    p1, p2 = random.sample(people, 2)

    img1 = os.path.join(dataset_path, p1, os.listdir(os.path.join(dataset_path, p1))[0])
    img2 = os.path.join(dataset_path, p2, os.listdir(os.path.join(dataset_path, p2))[0])

    emb1 = get_embedding(img1)
    emb2 = get_embedding(img2)

    diff_distances.append(np.linalg.norm(emb1 - emb2))

threshold = (np.mean(same_distances) + np.mean(diff_distances)) / 2

# 🔥 FIXED (not too strict)
threshold = threshold - 0.02

print("Final Threshold:", threshold)


# =========================
# 10. MATCH FUNCTION
# =========================
def is_same_person(emb1, emb2):
    dist = np.linalg.norm(emb1 - emb2)
    return dist < threshold


# =========================
# 11. TRACKING FUNCTION
# =========================
def track_person(reference_img, frames):
    ref_emb = get_embedding(reference_img)

    detected = False
    entry = None
    exit = None
    match_count = 0

    for i, frame in enumerate(frames):
        emb = get_embedding(frame)

        if emb is None:
            continue

        if is_same_person(ref_emb, emb):
            match_count += 1

            if match_count >= 2 and not detected:
                entry = i - 1
                detected = True

        else:
            if detected:
                exit = i - 1
                break

            match_count = 0

    return entry, exit


# =========================
# 12. SIMULATED VIDEO
# =========================
frames = [
    "/content/train/train/brad/Brad Pitt_9.jpg",
    "/content/train/train/hrithik/Hrithik Roshan_50.jpg",
    "/content/train/train/hrithik/Hrithik Roshan_7.jpg",
    "/content/train/train/brad/Brad Pitt_0.jpg"
]


# =========================
# 13. FINAL EXECUTION
# =========================
ref = "/content/train/train/hrithik/Hrithik Roshan_50.jpg"

entry, exit = track_person(ref, frames)

fps = 1

# 🔥 SAFE EXECUTION
if entry is None or exit is None:
    print("Person not detected properly")
else:
    entry_time = entry / fps
    exit_time = exit / fps
    duration = exit_time - entry_time

    print("Entry Time:", entry_time)
    print("Exit Time:", exit_time)
    print("Duration:", duration)

Saving train-20260405T162434Z-3-001.zip to train-20260405T162434Z-3-001 (1).zip
Dataset folders: ['alia', 'hrithik', 'billie', 'vijay', 'brad']
Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 10s 475ms/step - accuracy: 0.5972 - loss: 1.0907
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 520ms/step - accuracy: 0.9097 - loss: 0.3514
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 480ms/step - accuracy: 0.9549 - loss: 0.1761
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 635ms/step - accuracy: 0.9653 - loss: 0.1422
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 483ms/step - accuracy: 0.9931 - loss: 0.0749
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 471ms/step - accuracy: 0.9965 - loss: 0.0382
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 634ms/step - accuracy: 1.0000 - loss: 0.0287
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 481ms/step - accuracy: 1.0000 - loss: 0.0213
Epoch 9/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 591ms/step - accuracy: 1.0000 - loss: 0.0173
Epoch 10/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 10s 501ms/step - accuracy: 1.0000 - loss: 0.0131
1/1 ━━━━━━━━━